In [2]:

from pathlib import Path
import json

def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
JSON_PATH = PROJECT_ROOT / "outputs" / "structured" / "document_structure.json"

data = json.loads(JSON_PATH.read_text(encoding="utf-8"))

tables = data.get("tables", [])

print(f"Tables found in JSON: {len(tables)}")

for index, table in enumerate(tables[:5], start=1):
    print(f"\n{'=' * 20} TABLE {index} {'=' * 20}")

    # Print the metadata without dumping the entire table.
    print("Top-level keys:", list(table.keys()))

    for key in ("label", "caption", "prov", "data", "num_rows", "num_cols"):
        if key in table:
            value = table[key]
            if key == "data":
                print("data:", str(value)[:500])
            else:
                print(f"{key}: {value}")


Tables found in JSON: 12

==================== TABLE 1 ====================
Top-level keys: ['self_ref', 'parent', 'children', 'content_layer', 'label', 'prov', 'captions', 'references', 'footnotes', 'data', 'annotations']
label: table
prov: [{'page_no': 1, 'bbox': {'l': 34.17043790394828, 't': 713.5133628138453, 'r': 277.44794913067915, 'b': 686.3731652107148, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 0]}]
data: {'table_cells': [{'bbox': {'l': 36.0, 't': 130.418, 'r': 109.52100000000002, 'b': 139.0562568421052, 'coord_origin': 'TOPLEFT'}, 'row_span': 1, 'col_span': 1, 'start_row_offset_idx': 0, 'end_row_offset_idx': 1, 'start_col_offset_idx': 0, 'end_col_offset_idx': 1, 'text': 'BSE CODE: 500400', 'column_header': False, 'row_header': False, 'row_section': False, 'fillable': False}, {'bbox': {'l': 180.02, 't': 130.418, 'r': 276.24199999999996, 'b': 139.0562568421052, 'coord_origin': 'TOPLEFT'}, 'row_spa

==================== TABLE 2 ====================
Top-level keys: ['self_ref

In [3]:


for table_number, table in enumerate(tables[:5], start=1):
    provenance = table.get("prov", [{}])[0]
    table_data = table.get("data", {})
    cells = table_data.get("table_cells", [])

    print(f"\n{'=' * 18} TABLE {table_number} {'=' * 18}")
    print("Page:", provenance.get("page_no"))
    print("Table bounding box:", provenance.get("bbox"))
    print("Cell count:", len(cells))

    for cell in cells:
        print(
            f"row={cell.get('start_row_offset_idx')}, "
            f"col={cell.get('start_col_offset_idx')}, "
            f"row_span={cell.get('row_span')}, "
            f"col_span={cell.get('col_span')}, "
            f"text={cell.get('text')!r}"
        )



================== TABLE 1 ==================
Page: 1
Table bounding box: {'l': 34.17043790394828, 't': 713.5133628138453, 'r': 277.44794913067915, 'b': 686.3731652107148, 'coord_origin': 'BOTTOMLEFT'}
Cell count: 4
row=0, col=0, row_span=1, col_span=1, text='BSE CODE: 500400'
row=0, col=1, row_span=1, col_span=1, text='NSE CODE: TATAPOWER'
row=1, col=0, row_span=1, col_span=1, text='Bloomberg CODE: TPWR:IN'
row=1, col=1, row_span=1, col_span=1, text='SENSEX: 67,127'

================== TABLE 2 ==================
Page: 1
Table bounding box: {'l': 372.6753274638541, 't': 647.6006195546781, 'r': 566.2624924873506, 'b': 381.783394227379, 'coord_origin': 'BOTTOMLEFT'}
Cell count: 69
row=0, col=0, row_span=1, col_span=4, text='Company Data'
row=1, col=0, row_span=1, col_span=3, text='Market Cap (cr)'
row=1, col=3, row_span=1, col_span=1, text='Rs. 86,856'
row=2, col=0, row_span=1, col_span=3, text='Enterprise Value (cr)'
row=2, col=3, row_span=1, col_span=1, text='Rs. 131,339'
row=3, col=0

In [5]:
import csv
JSON_PATH = PROJECT_ROOT / "outputs" / "structured" / "document_structure.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structured" / "tables"

data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
table = data["tables"][1]  # Table 2: Company Data

cells = table["data"]["table_cells"]

# Determine the required grid size from the cell positions and spans.
row_count = max(
    cell["start_row_offset_idx"] + cell["row_span"]
    for cell in cells
)
col_count = max(
    cell["start_col_offset_idx"] + cell["col_span"]
    for cell in cells
)

grid = [["" for _ in range(col_count)] for _ in range(row_count)]

for cell in cells:
    row = cell["start_row_offset_idx"]
    col = cell["start_col_offset_idx"]
    text = cell.get("text", "").strip()

    # Store each cell's text only at its starting position.
    # Do not repeat it into the columns covered by its span.
    grid[row][col] = text

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "table_002_cells.csv"

with output_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    writer.writerows(grid)

print(f"Saved: {output_path}")
print(f"Grid size: {row_count} rows x {col_count} columns")
print("\nPreview:")
for row in grid[:12]:
    print(row)


Saved: D:\Acalvio\DeCraft\outputs\structured\tables\table_002_cells.csv
Grid size: 23 rows x 4 columns

Preview:
['Company Data', '', '', '']
['Market Cap (cr)', '', '', 'Rs. 86,856']
['Enterprise Value (cr)', '', '', 'Rs. 131,339']
['Outstanding Shares (cr)', '', '', '319.6']
['Free Float', '', '', '53.0%']
['Dividend Yield', '', '', '0.7%']
['52 week high', '', '', 'Rs. 277']
['52 week low', '', '', 'Rs. 182']
['6m average volume', '', '', '816,497']
['Beta', '', '', '1.0']
['Face value', '', '', 'Rs. 1']
['Shareholding %', 'Q3FY23', 'Q4FY23', 'Q1FY24']
